In [1]:
# from fiducial_detection import *
import cv2
import config
import os
from numba import njit
import numpy as np
import math
import flagger

In [2]:
cap = cv2.VideoCapture(0)

In [3]:
@njit(cache=True)
def getPixelLen_Corners(bits):
  # Calculates pixel length
  pixelLen = int(math.sqrt(bits))

  # Corners of square / fiducial marker
  corners = np.array([[0,0],[bits,0],[bits,bits],[0,bits]],np.float32)
  return pixelLen, corners
@njit(cache=True)
def getRes(im_er ,pixelLen):
  # Computes szie of im_er and size of each grid cell
  im_size=im_er.shape
  #print("Image Shape:",im_size)
  grid_cell_size = (im_size[0]//pixelLen, im_size[1]//pixelLen)
  #print("Grid Cell size:",grid_cell_size)

  # Loop and calculate mean intensity of each cell
  res = []
  for r in range(pixelLen):
    for c in range(pixelLen):
      y_1 = int(r * pixelLen )
      x_1 = int(c * pixelLen )
      y_2 = y_1 + grid_cell_size[1]
      x_2 = x_1 + grid_cell_size[0]
      cell = im_er[y_1:y_2,x_1:x_2]
      #cv2_imshow(cell)
      cell_val = cell.mean()/255

      if cell_val>0.3:
        res.append(1)
      else:
        res.append(0)
  return res


def get_contour_bits(im,cnt,bits):
  """ 
  bits = total number of bits in the fiducial. For RM it is 49 bits in total arranged in a 7x7 grid
  img = camera feed frame that contains the fiducial
  cnt = contour of the detected fiducial
  This function takes in the image, extracts the detected fiducial, coverts it into a rectangular image
  and gives out the signature of the detected fiducial
  """

  pixelLen,corners = getPixelLen_Corners(bits) 

  # Transformation Matrix that maps the contour points to corners
  M = cv2.getPerspectiveTransform(cnt, corners)

  # Apply transformation onto input image and store in variable
  dst = cv2.warpPerspective(im, M, (bits, bits),flags=cv2.INTER_LINEAR)
  #cv2.imshow(dst) 

  # Converst dst to greyscale
  gs =  cv2.cvtColor(dst, cv2.COLOR_BGR2GRAY)
  ret, im_thres = cv2.threshold(gs, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  #cv2.imshow(im_thres)

  # Create rectangular structuring object
  element = cv2.getStructuringElement(cv2.MORPH_RECT,(3,3))

  # Applies erosion element onto im_thres
  im_er = cv2.erode(im_thres,element)
  #cv2.imshow(im_er)

  res = getRes(im_er ,pixelLen)

  return res, dst

In [4]:
def load_marker_dictionary(marker,bits):
  
  # Variables to store binary signatature, width, and height of image
  sigs = []
  w = len(marker[0])
  h= len(marker)

  # im_marker = cv2.cvtColor(marker,cv2.COLOR_BGR2GRAY)
  # Creates copy and defines corner coordinates of countour
  im_marker = marker.copy()
  cnt = np.array([[0,0],[w,0],[w,h],[0,h]],np.float32)

  # Loops through each quandrant and calls "get_contour bits" function 
  for i in range(4):
    sig,_ = get_contour_bits(im_marker,cnt,bits)

    # Adds binary signature to list
    sigs.append((sig,i))

    # Rotates image 90°
    im_marker = cv2.rotate(im_marker, cv2.ROTATE_90_CLOCKWISE)

  return sigs

def loadMarkers(markers) -> dict:
      marker_dict = {}
      for marker in markers:
            marker_img = cv2.imread(marker['marker_file'])
            marker['sig'] = load_marker_dictionary(marker_img, bits=49)
            marker_dict[marker['marker_id']] = marker
      return marker_dict
markers =loadMarkers(config.fiducials['markers'])

In [5]:
fid_flagger= flagger.Flagger(
    config.gameplay['OPPONENT_TEAM_COLOR'], config.gameplay['HSV_LOWER_RED'], config.gameplay['HSV_UPPER_RED'], config.gameplay['HSV_LOWER_RED_2'], config.gameplay['HSV_UPPER_RED_2'], config.gameplay['HSV_LOWER_BLUE'], config.gameplay['HSV_UPPER_BLUE'])

In [6]:
@njit(cache=True)
def getMarkerMatch(image_signal,m_signal_tuple, threshold) -> tuple:
    m_signal = np.array(m_signal_tuple[0])
    m_orientation_index = m_signal_tuple[1]
    n_misses = np.abs(np.sum((m_signal-image_signal)*m_signal))
    if (m_signal.sum()-n_misses)>=int(threshold*m_signal.sum()):
      return (m_signal.sum()-n_misses, m_orientation_index)
    else:
      return (0,0)
  

@njit(cache=True)
def sortHits(hits):
    hits.sort(key = lambda x: x[1]) 
    return hits


def matchMarker(image_signal, threshold):
    hits = []
    for marker in markers.values():
      for m_signal_tuple in marker['sig']:
         sum_,index= getMarkerMatch(image_signal,m_signal_tuple, threshold)
         if sum_ >0:
           hits.append((marker['marker_id'],sum_, index))
    if len(hits)>0:
      hits = sortHits(hits)
    #print('hits', hits)
    return hits

In [7]:
@njit(fastmath=True,nogil=True, cache=True, parallel=True)
def order_contour(cnt):
  # Calculates center point of contour through average of 4 points
  cx = (cnt[0][0][0] + cnt[1][0][0] + cnt[2][0][0] + cnt[3][0][0])/4.0
  cy = (cnt[0][0][1] + cnt[1][0][1] + cnt[2][0][1] + cnt[3][0][1])/4.0

  # Orders the contour
  if(cnt[0][0][0] <= cx and cnt[0][0][1]<=cy):
    temp = cnt[3].copy()
    cnt[3] = cnt[1]
    cnt[1] = temp
  else:
    temp = cnt[0].copy()
    cnt[0] = cnt[1]
    cnt[1] = temp

    temp2 = cnt[3].copy()
    cnt[3] = cnt[2]
    cnt[2] = temp2

  return cnt

In [8]:
def findMarker(img, threshold):

    bits = 49
    orig_img = img.copy()
    # Converts img to greyscale
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Binary Image with white regions for areas of interest
    img_thresh = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 11, 5)

    # Finds contours in binary image
    contours,_ = cv2.findContours(img_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    res_contours = []

    # Displays original image if no contours found 
    if len(contours)==0:
      cv2.imshow('ROBOMASTER', orig_img)
      cv2.waitKey(20)

      return [],[],None,'', ''
    

    for cnt in contours:

      # Simplify Contour
      poly_cnt = cv2.approxPolyDP(cnt, 0.01*cv2.arcLength(cnt,True), True)
      
      # Checks if poly_cnt has 4 vertices, is convex, and has area greater than 1/1024
      if(len(poly_cnt)!=4 or not(cv2.isContourConvex(poly_cnt)) or cv2.contourArea(poly_cnt)<int((1/1024)*(img_thresh.shape[0]*img_thresh.shape[1])) ) :
        continue
      
      # Converts contoour points to float32 type
      poly_cnt = np.float32(poly_cnt)

      # Specifies termination criteria
      criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_COUNT, 40, 0.01)

      # Refine corner of contour 
      poly_cnt = cv2.cornerSubPix(img_thresh, poly_cnt, (5,5), (-1,-1),criteria)

      # Orders countour points 
      poly_cnt = order_contour(poly_cnt)

      # Adds to list
      res_contours.append(np.int32(poly_cnt.reshape((4,2))))

    # Check if contour successfully detected
    if len(res_contours)==0:
      cv2.imshow('ROBOMASTER', orig_img)
      cv2.waitKey(20)
      return [],[],None,'', ''
    
    # If multiple contour detected prints message that only first will be used
    if len(res_contours)>1:
      print('{} contours detected, considering only the first'.format(len(res_contours)))
    
    # Converts to ordered and refined contour points of first contour as float32
    res_contours = np.float32(res_contours[0])

    # Converts to NumPy array
    image_signal, warped_image = get_contour_bits(orig_img, res_contours, bits)
    image_signal = np.array(image_signal)

    # Determine color of marker
    detected_color = fid_flagger.red_or_blue(warped_image)

    # Compares marker to marker patterns
    hits = matchMarker(image_signal, threshold)

    # Checks if any mathces
    if len(hits)>0:
      detected_marker_char = hits[-1][0]
    else:
      detected_marker_char = ''

    # Set result image to the input image
    res_image = img

    res_image = cv2.drawContours(orig_img, [np.int64(res_contours)],-1,(255,255,0),3)
    font = cv2.FONT_HERSHEY_SIMPLEX  
    fontScale = 10
    color = (255, 0, 0)
    thickness = 2
  
    res_image = cv2.putText(res_image,detected_marker_char,(200,250) ,font, fontScale, color, thickness, cv2.LINE_AA)
    cv2.imshow('ROBOMASTER', res_image)
    return hits, res_contours, res_image, detected_color, detected_marker_char

In [11]:
# clear all cv2 windows
cv2.destroyAllWindows()

In [12]:
import time

# Initialize variables for FPS calculation
fps_start_time = time.time()
fps_frame_counter = 0

while True:
    # get a frame and display
    ret, frame = cap.read()
    if not ret:
        pass
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    return_ = findMarker(frame, 0.7)
    if return_ != ([], [], None, ''):
        _, _, _, marker_color, marker_char = return_
        os.system('clear')
        if marker_char != '':
            print(marker_char, marker_color)

    # Calculate FPS
    fps_frame_counter += 1
    if time.time() - fps_start_time >= 1:
        fps = fps_frame_counter / (time.time() - fps_start_time)
        fps_start_time = time.time()
        fps_frame_counter = 0
        print("FPS:", round(fps, 2))

FPS: 8.37
FPS: 8.97
FPS: 8.27
B RED
3 contours detected, considering only the first
D RED
3 contours detected, considering only the first
D RED
2 contours detected, considering only the first
D RED
FPS: 8.91
2 contours detected, considering only the first
B RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
7 contours detected, considering only the first
B RED
7 contours detected, considering only the first
B RED
7 contours detected, considering only the first
B RED
7 contours detected, considering only the first
B RED
7 contours detected, considering only the first
B RED
6 contours detected, considering only the first
B RED
FPS: 10.88
6 contours detected, considering only the first
B RED
6 contours detected, considering only the first
B RED
7 contours detected, considering only the first
B RED
7 contours detected, considering only the first
B RED
7 contours detected, cons

c:\Users\Samsepi0l\miniconda3\envs\CVRM\lib\site-packages\ipykernel_launcher.py:16: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  


FPS: 18.98
D RED
FPS: 22.35
FPS: 17.8


In [10]:
# while(True):
#     # get a frame and display
#       ret, frame = cap.read()
#       if not ret:
#             pass
#       if cv2.waitKey(1) & 0xFF == ord('q'):
#             break
     
#       return_ = findMarker(frame,0.7)
#       if return_!=([],[],None,''):
#             _,_,_,marker_color,marker_char = return_
#             os.system('clear')
#             if marker_char!='':
#                   print(marker_char,marker_color)

5 contours detected, considering only the first


C:\Temp\ipykernel_6128\1638531198.py:14: NumbaPendingDeprecationWarning: 
Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'lst' of function 'ol_list_sort.<locals>.impl'.

For more information visit https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types

File "..\..\..\..\miniconda3\envs\CVRM\lib\site-packages\numba\cpython\listobj.py", line 1102:

    def impl(lst, key=None, reverse=False):
    ^

  hits.sort(key = lambda x: x[1])
c:\Users\Samsepi0l\miniconda3\envs\CVRM\lib\site-packages\numba\core\ir_utils.py:2147: NumbaPendingDeprecationWarning: 
Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'hits' of function 'sortHits'.

For more information visit https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types

File "..\..\..\..\..\..\Temp\ipykernel_6128\16

B RED
6 contours detected, considering only the first
A RED
8 contours detected, considering only the first
B RED
7 contours detected, considering only the first
A RED
7 contours detected, considering only the first
A RED
8 contours detected, considering only the first
B RED
8 contours detected, considering only the first
B RED
7 contours detected, considering only the first
A RED
7 contours detected, considering only the first
A RED
6 contours detected, considering only the first
A RED
7 contours detected, considering only the first
A RED
7 contours detected, considering only the first
A RED
7 contours detected, considering only the first
A RED
8 contours detected, considering only the first
B RED
8 contours detected, considering only the first
B RED
7 contours detected, considering only the first
A RED
8 contours detected, considering only the first
A RED
8 contours detected, considering only the first
A RED
6 contours detected, considering only the first
A RED
7 contours detected, c

c:\Users\Samsepi0l\miniconda3\envs\CVRM\lib\site-packages\ipykernel_launcher.py:10: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  # Remove the CWD from sys.path while we load stuff.


D RED
6 contours detected, considering only the first
D RED
5 contours detected, considering only the first
D RED
3 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
B RED
6 contours detected, considering only the first
B RED
5 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
B RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
D RED
6 contours detected, considering only the first
B RED
6 contours detected, considering only the first
B RED
6 contours detected, considering only the first
B RED
6 contours detected, considering only the first
D RED
6 contours detected, c